# 02 - Limpeza e Enriquecimento dos Dados de Renda por Setor Censitário

Extrai e limpa os dados de renda do IBGE, filtra por Recife e faz join com o shapefile de setores.

In [12]:
import zipfile
import os
import glob
import pandas as pd
import geopandas as gpd
from IPython.display import display

In [13]:
from pathlib import Path

def find_root():
    for p in [Path.cwd(), *Path.cwd().parents]:
        if (p / '.git').exists():
            return p
    return Path.cwd()

ROOT        = find_root()
ZIP_PATH    = ROOT / 'data' / 'raw' / 'renda_ibge' / 'Agregados_por_setores_renda_responsavel_BR_20260508_csv.zip'
EXTRACT_DIR = ROOT / 'data' / 'raw' / 'renda_ibge' / 'extraido'
GEO_PATH    = ROOT / 'data' / 'raw' / 'setores_ibge' / 'recife_setores.geojson'
OUTPUT_DIR  = ROOT / 'data' / 'processed'
OUTPUT_PATH = OUTPUT_DIR / 'recife_renda.geojson'

os.makedirs(ZIP_PATH.parent, exist_ok=True)
os.makedirs(EXTRACT_DIR, exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f'ROOT: {ROOT}')
print(f'Diretório de extração: {EXTRACT_DIR}')
print(f'Diretório de saída:    {OUTPUT_DIR}')

ROOT: c:\Users\rodri\OneDrive\Documents\ProjetosPessoais\IVU-RECIFE
Diretório de extração: c:\Users\rodri\OneDrive\Documents\ProjetosPessoais\IVU-RECIFE\data\raw\renda_ibge\extraido
Diretório de saída:    c:\Users\rodri\OneDrive\Documents\ProjetosPessoais\IVU-RECIFE\data\processed


In [14]:
# Extrai o arquivo ZIP local com os dados de renda do IBGE
csv_existente = glob.glob(os.path.join(EXTRACT_DIR, '**', '*.csv'), recursive=True)

if not csv_existente:
    if not ZIP_PATH.exists():
        raise FileNotFoundError(f'ZIP de renda não encontrado em: {ZIP_PATH}')

    if not zipfile.is_zipfile(ZIP_PATH):
        raise zipfile.BadZipFile(f'Arquivo ZIP inválido ou corrompido: {ZIP_PATH}')

    print('Extraindo ZIP de renda...')
    with zipfile.ZipFile(ZIP_PATH, 'r') as zf:
        zf.extractall(EXTRACT_DIR)
    print(f'Arquivos extraídos em: {EXTRACT_DIR}')
else:
    print(f'Usando CSV de renda já extraído em: {EXTRACT_DIR}')

print('Conteúdo:', os.listdir(EXTRACT_DIR))

Usando CSV de renda já extraído em: c:\Users\rodri\OneDrive\Documents\ProjetosPessoais\IVU-RECIFE\data\raw\renda_ibge\extraido
Conteúdo: ['Agregados_por_setores_renda_responsavel_BR.csv']


In [15]:
# Localiza o arquivo CSV dentro do diretório extraído (busca recursiva)
csv_files = glob.glob(os.path.join(EXTRACT_DIR, '**', '*.csv'), recursive=True)
if not csv_files:
    raise FileNotFoundError('Nenhum arquivo .csv encontrado após extração.')

CSV_PATH = csv_files[0]
print(f'CSV encontrado: {CSV_PATH}')
if len(csv_files) > 1:
    print(f'  (outros CSVs encontrados: {csv_files[1:]})')

CSV encontrado: c:\Users\rodri\OneDrive\Documents\ProjetosPessoais\IVU-RECIFE\data\raw\renda_ibge\extraido\Agregados_por_setores_renda_responsavel_BR.csv


In [16]:
# Carrega o CSV com pandas; tenta separadores comum do IBGE (;)
print('Carregando CSV de renda...')
try:
    df = pd.read_csv(CSV_PATH, sep=';', dtype=str, encoding='latin-1')
    print(f'Linhas carregadas: {len(df)}')
    print('Colunas:', df.columns.tolist())
except Exception as e:
    print(f'ERRO ao carregar CSV: {e}')
    raise

print('\nPrimeiras linhas:')
display(df.head())

Carregando CSV de renda...
Linhas carregadas: 458772
Colunas: ['CD_SETOR', 'V06001', 'V06002', 'V06003', 'V06004', 'V06005', 'V06006']

Primeiras linhas:


,CD_SETOR,V06001,V06002,V06003,V06004,V06005,V06006
0,110001505000002,336,928,1.67,2453.03,13047638.93,1212
1,110001505000003,208,556,1.79,2126.44,2348222.39,1500
2,110001505000004,85,222,1.6,1664.94,1270082.06,1200
3,110001505000006,281,783,2.41,1722.4,1446560.6,1200
4,110001505000007,291,748,1.38,1930.94,2255368.87,1212


In [17]:
# Identifica a coluna de código do setor censitário (tipicamente 'Cod_setor' ou 'CD_GEOCODI')
COD_RECIFE = '2611606'

candidatas = [c for c in df.columns if 'setor' in c.lower() or 'geocod' in c.lower() or 'cod' in c.lower()]
print(f'Colunas candidatas para código do setor: {candidatas}')

# Usa a primeira candidata encontrada; ajuste aqui caso necessário
COL_SETOR = candidatas[0] if candidatas else df.columns[0]
print(f'Usando coluna: {COL_SETOR}')

Colunas candidatas para código do setor: ['CD_SETOR']
Usando coluna: CD_SETOR


In [18]:
# Filtra linhas cujos primeiros 7 dígitos do código do setor correspondem a Recife
print(f'Filtrando setores de Recife (primeiros 7 dígitos == {COD_RECIFE})...')
df[COL_SETOR] = df[COL_SETOR].astype(str).str.strip()
df_recife = df[df[COL_SETOR].str[:7] == COD_RECIFE].copy()
print(f'Setores de renda de Recife encontrados: {len(df_recife)}')
display(df_recife.head())

Filtrando setores de Recife (primeiros 7 dígitos == 2611606)...
Setores de renda de Recife encontrados: 2814


,CD_SETOR,V06001,V06002,V06003,V06004,V06005,V06006
110633,261160605180001,346,693,1.53,5054.18,30184825.2,3100
110634,261160605180002,220,529,1.75,5567.78,55810857.86,4500
110635,261160605180003,237,472,1.25,3006.58,4796567.18,2300
110636,261160605180004,244,496,1.21,4081.47,18070460.02,3000
110637,261160605180005,292,535,1.03,3194.82,6789991.04,2400


In [19]:
# Carrega o GeoJSON de setores de Recife gerado no notebook 01
print(f'Carregando GeoJSON de setores: {GEO_PATH}')
try:
    gdf = gpd.read_file(GEO_PATH)
    print(f'Setores no GeoJSON: {len(gdf)}')
    print('Colunas do GeoJSON:', gdf.columns.tolist())
    display(gdf.head(3))
except FileNotFoundError:
    print(f'ERRO: GeoJSON não encontrado em {GEO_PATH}')
    print('Execute o notebook 01_coleta_shapefile.ipynb primeiro.')
    raise
except Exception as e:
    print(f'ERRO ao carregar GeoJSON: {e}')
    raise

Carregando GeoJSON de setores: c:\Users\rodri\OneDrive\Documents\ProjetosPessoais\IVU-RECIFE\data\raw\setores_ibge\recife_setores.geojson
Setores no GeoJSON: 2835
Colunas do GeoJSON: ['CD_SETOR', 'SITUACAO', 'CD_SIT', 'CD_TIPO', 'AREA_KM2', 'CD_REGIAO', 'NM_REGIAO', 'CD_UF', 'NM_UF', 'CD_MUN', 'NM_MUN', 'CD_DIST', 'NM_DIST', 'CD_SUBDIST', 'NM_SUBDIST', 'CD_BAIRRO', 'NM_BAIRRO', 'CD_NU', 'NM_NU', 'CD_FCU', 'NM_FCU', 'CD_AGLOM', 'NM_AGLOM', 'CD_RGINT', 'NM_RGINT', 'CD_RGI', 'NM_RGI', 'CD_CONCURB', 'NM_CONCURB', 'geometry']


,CD_SETOR,SITUACAO,CD_SIT,CD_TIPO,AREA_KM2,CD_REGIAO,NM_REGIAO,CD_UF,NM_UF,CD_MUN,...,NM_FCU,CD_AGLOM,NM_AGLOM,CD_RGINT,NM_RGINT,CD_RGI,NM_RGI,CD_CONCURB,NM_CONCURB,geometry
0,261160605180001,Urbana,1,0,0.054228,2,Nordeste,26,Pernambuco,2611606,...,None,None,None,2601,Recife,260001,Recife,2611606,Recife/PE,"POLYGON ((-34.88768 -8.06219, -34.88783 -8.062..."
1,261160605180002,Urbana,1,0,0.105716,2,Nordeste,26,Pernambuco,2611606,...,None,None,None,2601,Recife,260001,Recife,2611606,Recife/PE,"POLYGON ((-34.88701 -8.06055, -34.88671 -8.060..."
2,261160605180003,Urbana,1,0,0.056487,2,Nordeste,26,Pernambuco,2611606,...,None,None,None,2601,Recife,260001,Recife,2611606,Recife/PE,"POLYGON ((-34.8858 -8.0599, -34.88443 -8.06056..."


In [20]:
# Identifica a coluna de código do setor no GeoJSON (tipicamente 'CD_SETOR')
geo_candidatas = [c for c in gdf.columns if 'setor' in c.lower() or 'geocod' in c.lower()]
print(f'Colunas candidatas no GeoJSON: {geo_candidatas}')

COL_GEO_SETOR = geo_candidatas[0] if geo_candidatas else 'CD_SETOR'
print(f'Usando coluna do GeoJSON: {COL_GEO_SETOR}')

# Garante que ambos os códigos estão no mesmo formato de string
gdf[COL_GEO_SETOR]   = gdf[COL_GEO_SETOR].astype(str).str.strip()
df_recife[COL_SETOR] = df_recife[COL_SETOR].astype(str).str.strip()

Colunas candidatas no GeoJSON: ['CD_SETOR']
Usando coluna do GeoJSON: CD_SETOR


In [21]:
# Faz o join entre o GeoDataFrame (geometria) e o DataFrame de renda pelo código do setor
print('Realizando join por código do setor censitário...')
gdf_renda = gdf.merge(
    df_recife,
    left_on=COL_GEO_SETOR,
    right_on=COL_SETOR,
    how='left'
)

setores_com_renda = gdf_renda[COL_SETOR].notna().sum()
print(f'Total de setores após join: {len(gdf_renda)}')
print(f'Setores com dados de renda: {setores_com_renda}')
print(f'Setores sem correspondência: {len(gdf_renda) - setores_com_renda}')
display(gdf_renda.head(3))

Realizando join por código do setor censitário...
Total de setores após join: 2835
Setores com dados de renda: 2835
Setores sem correspondência: 0


,CD_SETOR,SITUACAO,CD_SIT,CD_TIPO,AREA_KM2,CD_REGIAO,NM_REGIAO,CD_UF,NM_UF,CD_MUN,...,NM_RGI,CD_CONCURB,NM_CONCURB,geometry,V06001,V06002,V06003,V06004,V06005,V06006
0,261160605180001,Urbana,1,0,0.054228,2,Nordeste,26,Pernambuco,2611606,...,Recife,2611606,Recife/PE,"POLYGON ((-34.88768 -8.06219, -34.88783 -8.062...",346,693,1.53,5054.18,30184825.2,3100
1,261160605180002,Urbana,1,0,0.105716,2,Nordeste,26,Pernambuco,2611606,...,Recife,2611606,Recife/PE,"POLYGON ((-34.88701 -8.06055, -34.88671 -8.060...",220,529,1.75,5567.78,55810857.86,4500
2,261160605180003,Urbana,1,0,0.056487,2,Nordeste,26,Pernambuco,2611606,...,Recife,2611606,Recife/PE,"POLYGON ((-34.8858 -8.0599, -34.88443 -8.06056...",237,472,1.25,3006.58,4796567.18,2300


In [22]:
# Salva o resultado enriquecido como GeoJSON em data/processed/
print(f'Salvando resultado em: {OUTPUT_PATH}')
try:
    gdf_renda.to_file(OUTPUT_PATH, driver='GeoJSON')
    print('Arquivo salvo com sucesso.')
    print(f'Linhas: {len(gdf_renda)} | Colunas: {len(gdf_renda.columns)}')
except Exception as e:
    print(f'ERRO ao salvar GeoJSON: {e}')
    raise

Salvando resultado em: c:\Users\rodri\OneDrive\Documents\ProjetosPessoais\IVU-RECIFE\data\processed\recife_renda.geojson
Arquivo salvo com sucesso.
Linhas: 2835 | Colunas: 36
